In [6]:
import os
import gzip
import struct
import shutil
from glob import glob
from pathlib import Path

import numpy as np
from PIL import Image

# --- ПАРАМЕТРЫ ---
ARCHIVE_DIR = "archive/emnist_source_files"   # ← сюда распакуйте archive.zip
OUTPUT_DIR  = "output"                     # ← куда будут сохраняться PNG
MAX_PER_SPLIT = 1000                       # ← сколько файлов сохранять из каждого набора
# ------------------

# считываем mapping.txt (label → символ), если он есть
def load_mapping(txt_path):
    mapping = {}
    with open(txt_path, 'r') as f:
        for line in f:
            label, charcode = line.strip().split()
            mapping[int(label)] = chr(int(charcode))
    return mapping

# загрузка изображений из IDX-файла
def load_idx_images(path):
    # поддерживает .gz и «сырой» .idx
    opener = gzip.open if str(path).endswith(".gz") else open
    with opener(path, "rb") as f:
        # 4 байта magic, 4 — число, 4 — rows, 4 — cols
        magic, num, rows, cols = struct.unpack(">IIII", f.read(16))
        data = np.frombuffer(f.read(), dtype=np.uint8)
    return data.reshape(num, rows, cols)

# загрузка меток
def load_idx_labels(path):
    opener = gzip.open if str(path).endswith(".gz") else open
    with opener(path, "rb") as f:
        magic, num = struct.unpack(">II", f.read(8))
        data = np.frombuffer(f.read(), dtype=np.uint8)
    return data

# основная функция: находит все пары и сохраняет
def extract_all(archive_dir, output_dir, max_per_split=1000):
    os.makedirs(output_dir, exist_ok=True)
    # найдём все изображения
    img_files = sorted(glob(os.path.join(archive_dir, "*-images-idx3-ubyte*")))
    for img_path in img_files:
        base = Path(img_path).name.replace("-images-idx3-ubyte", "")
        lbl_path = os.path.join(archive_dir, base + "-labels-idx1-ubyte")
        if not os.path.exists(lbl_path) and not os.path.exists(lbl_path + ".gz"):
            print(f"Label file для {base} не найден, пропускаем.")
            continue

        # определим, к какому «сплиту» относится этот набор
        # base примеры: 'emnist-digits-train', 'emnist-letters-test', 'emnist-balanced-train' и т.п.
        parts = base.split("-")
        emnist, split, subset = parts[0], parts[1], parts[2]  # emnist, digits/letters/... , train/test
        out_subdir = os.path.join(output_dir, subset, split)
        os.makedirs(out_subdir, exist_ok=True)

        images = load_idx_images(img_path)
        labels = load_idx_labels(lbl_path + (".gz" if not os.path.exists(lbl_path) else ""))

        # попробуем загрузить mapping (если есть)
        map_fname = os.path.join(archive_dir, f"{emnist}-{split}-mapping.txt")
        mapping = {}
        if os.path.exists(map_fname):
            try:
                mapping = load_mapping(map_fname)
            except Exception as e:
                print("Не удалось прочитать mapping:", e)

        for i, (img, lbl) in enumerate(zip(images, labels)):
            if i >= max_per_split:
                break
            im = Image.fromarray(img, mode='L')
            # подпишем файл «digit_00001.png» или «A_00001.png», если mapping есть
            label_str = mapping.get(int(lbl), lbl)
            fname = f"{label_str}_{i:05d}.png"
            im.save(os.path.join(out_subdir, fname))
        print(f"Сохранено {min(len(images),max_per_split)} файлов в {out_subdir}")

# если у вас archive.zip, сначала распакуйте:
# shutil.unpack_archive("archive.zip", "archive_unzipped")
extract_all("archive_unzipped", OUTPUT_DIR, MAX_PER_SPLIT)


In [3]:
import os
from glob import glob
import numpy as np
import pandas as pd
from PIL import Image

# ← поменяли на папку, где реально лежат CSV
INPUT_DIR    = "archive"      
OUTPUT_DIR   = "output"
MAX_PER_SPLIT = 1000

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Ищем все emnist-<subset>-<split>.csv в archive/
for csv_path in glob(os.path.join(INPUT_DIR, "emnist-*-*.csv")):
    name = os.path.basename(csv_path).replace(".csv","")
    _, subset, split = name.split("-")      # например ['emnist','digits','train']
    out_subdir = os.path.join(OUTPUT_DIR, subset, split)
    os.makedirs(out_subdir, exist_ok=True)

    # читаем CSV: первая колонка — метка, остальные 784 — пиксели
    df = pd.read_csv(csv_path, header=None)
    labels = df.iloc[:,0].values.astype(int)
    pixels = df.iloc[:,1:].values.astype(np.uint8)

    n = min(len(pixels), MAX_PER_SPLIT)
    for i in range(n):
        arr = pixels[i].reshape(28,28)
        im = Image.fromarray(arr, mode="L")
        fname = f"{labels[i]}_{i:05d}.png"
        im.save(os.path.join(out_subdir, fname))

    print(f"[{name}] сохранил {n} PNG в {out_subdir}")


[emnist-letters-test] сохранил 1000 PNG в output/letters/test
[emnist-mnist-train] сохранил 1000 PNG в output/mnist/train
[emnist-mnist-test] сохранил 1000 PNG в output/mnist/test
[emnist-byclass-train] сохранил 1000 PNG в output/byclass/train
[emnist-letters-train] сохранил 1000 PNG в output/letters/train
[emnist-digits-train] сохранил 1000 PNG в output/digits/train
[emnist-balanced-train] сохранил 1000 PNG в output/balanced/train
[emnist-digits-test] сохранил 1000 PNG в output/digits/test
[emnist-bymerge-train] сохранил 1000 PNG в output/bymerge/train
[emnist-bymerge-test] сохранил 1000 PNG в output/bymerge/test
[emnist-byclass-test] сохранил 1000 PNG в output/byclass/test
[emnist-balanced-test] сохранил 1000 PNG в output/balanced/test
